In [2]:
# -*- coding: utf-8 -*-
"""
TabPFN + SHAP(shapiq) · 原始列层面解释（带可视化进度与阶段耗时 + 稳健修复）
依赖：
    pip install tabpfn shapiq pandas scikit-learn numpy torch tqdm
"""

import os
import numpy as np
import pandas as pd
import torch
from time import perf_counter
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from tqdm import tqdm

# --- TabPFN 兼容导入（官方包优先，其次 extensions） ---
try:
    from tabpfn import TabPFNClassifier  # 官方
except Exception:
    from tabpfn_extensions import TabPFNClassifier  # 你的 extensions 版本

import shapiq  # pip install shapiq


# ===================== 用户参数 =====================
CSV_PATH    = r"E:\adni_dataset\ADNI_Tabel.csv"
LABEL_COL   = "Group"
CLASSES     = ["SMCI", "PMCI"]       # 目标二分类标签
START_COL   = 3                      # 原始特征起始列（含）
TEST_SIZE   = 0.2
RANDOM_SEED = 42

# SHAP 参数（先小一点，确认流程 OK 再调大）
EXPLAIN_MAX_EVAL = 90               # 从测试集中抽多少样本做解释
SHAP_BUDGET      = 256               # Shapley 预算（子集采样预算，越大越稳越慢）
TOPK_PRINT       = 30
OUT_CSV          = "tabpfn_shap_importance_raw.csv"
OUT_CSV_PARTIAL  = "tabpfn_shap_importance_raw_partial.csv"  # 增量保存文件
SAVE_EVERY       = 32                # 每处理多少样本增量保存一次

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def section(msg: str):
    print("\n" + "=" * 12 + f" {msg} " + "=" * 12, flush=True)


def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    t0 = perf_counter()

    # ===================== 读取&清洗 =====================
    section("数据读取与清洗")
    t = perf_counter()
    df = pd.read_csv(CSV_PATH)

    # 原始特征列
    raw_cols = list(df.columns[START_COL:])

    # 标签清洗：统一去空格
    labels_raw = df[LABEL_COL].astype(str).str.strip()

    # 映射
    mapping = {CLASSES[0]: 0, CLASSES[1]: 1}
    y_map = labels_raw.map(mapping)

    # 二次“大小写无关”兜底
    if y_map.isna().any():
        lower_map = {CLASSES[0].lower(): 0, CLASSES[1].lower(): 1}
        y_map2 = labels_raw.str.lower().map(lower_map)
        y_map = y_map.fillna(y_map2)

    # 未映射的提示并剔除
    if y_map.isna().any():
        unmapped_vals = labels_raw[y_map.isna()].value_counts()
        print("[warn] 发现未映射到的标签值（将被剔除）：")
        print(unmapped_vals.to_string())
        print("[hint] 如需包含它们，请调整 CLASSES 或预先过滤。")

    keep_mask = ~y_map.isna()
    dropped = (~keep_mask).sum()
    if dropped > 0:
        print(f"[info] 因标签未映射/缺失删除样本数：{dropped}")
    df = df.loc[keep_mask].reset_index(drop=True)
    y_all = y_map.loc[keep_mask].astype(int).values

    # 若只剩一个类别，直接报错提示
    if np.unique(y_all).size < 2:
        raise ValueError(f"仅剩单一类别：{np.unique(y_all)}。请检查 LABEL_COL/CLASSES 或过滤策略。")

    # 特征清洗：inf/-inf -> NaN -> 中位数插补
    X_df = df[raw_cols].copy()
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    imputer = SimpleImputer(strategy="median")
    X_all = imputer.fit_transform(X_df.values).astype(np.float32)

    # === 新增：过滤零方差特征（全局）===
    rng = np.ptp(X_all, axis=0)  # max-min
    nonconst_mask = rng > 1e-12  # 或用 X_all.std(axis=0) > 1e-12
    if nonconst_mask.sum() < len(nonconst_mask):
        removed = [c for c, keep in zip(raw_cols, nonconst_mask) if not keep]
        print(f"[info] 过滤零方差特征 {len(removed)} 列：{removed[:10]}{' ...' if len(removed)>10 else ''}")
    X_all = X_all[:, nonconst_mask]
    raw_cols = [c for c, keep in zip(raw_cols, nonconst_mask) if keep]

    # 断言无 NaN/Inf
    if not np.isfinite(X_all).all():
        raise ValueError("插补后仍存在 NaN/Inf，请检查数据。")

    print(f"[done] 用时 {(perf_counter()-t):.2f}s · 数据形状: X={X_all.shape}, y={y_all.shape[0]}")

    # ===================== 划分 =====================
    section("划分训练/测试集")
    t = perf_counter()
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
    )
    print(f"Train: {X_train.shape}, Test: {X_test.shape}, Pos rate(train)={y_train.mean():.3f}")
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # ===================== 训练 TabPFN =====================
    section("训练 TabPFN")
    t = perf_counter()
    model = TabPFNClassifier(device=DEVICE)
    model.fit(X_train, y_train)
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")

    # 评估
    section("评估 TabPFN")
    t = perf_counter()
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # 基础指标
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, pos_label=1, zero_division=0)  # PRE
    sen = recall_score(y_test, y_pred, pos_label=1, zero_division=0)     # SEN = Recall
    f1  = f1_score(y_test, y_pred, pos_label=1, zero_division=0)         # F1

    # 特异度（SPE）由混淆矩阵计算
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    spe = tn / (tn + fp) if (tn + fp) > 0 else 0.0                       # SPE = TNR

    # AUC
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = float("nan")

    print(f"[TabPFN] ACC={acc:.4f} PRE={pre:.4f} SEN={sen:.4f} SPE={spe:.4f} F1={f1:.4f} AUC={auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))
    print(f"[done] 用时 {(perf_counter()-t):.2f}s")


    # ===================== SHAP（原始列层面，带进度条） =====================
    section("SHAP（原始列 · TabPFNExplainer · 带进度）")
    t = perf_counter()
    explainer = shapiq.Explainer(
        model=model,
        data=X_train,          # 用训练集作为上下文
        labels=y_train,
        index="SV",            # Shapley values
        max_order=1,           # 一阶（单特征）
    )

    n_eval = min(EXPLAIN_MAX_EVAL, len(X_test))
    if n_eval <= 0:
        print("[warn] 测试样本不足，跳过 SHAP。")
        return

    X_explain = X_test[:n_eval]

    # 聚合 mean(|SHAP|) 所需
    abs_shap_sum = np.zeros(len(raw_cols), dtype=np.float64)

    # 可选：增量保存，避免长时间运行中断
    def save_partial(done):
        if done <= 0:
            return
        mean_abs_shap_partial = abs_shap_sum / done
        (pd.DataFrame({
            "feature": raw_cols,
            "mean_abs_shap_partial": mean_abs_shap_partial
        }).sort_values("mean_abs_shap_partial", ascending=False)
         .to_csv(OUT_CSV_PARTIAL, index=False, encoding="utf-8-sig"))

    print(f"[info] n_eval={n_eval}, budget={SHAP_BUDGET}")
    skipped = 0
    avg_t = None
    with tqdm(total=n_eval, desc="SHAP explaining", unit="sample", miniters=1, mininterval=0.1) as pbar:
        for i in range(n_eval):
            t1 = perf_counter()
            try:
                sv = explainer.explain(X_explain[i], budget=SHAP_BUDGET)
                # 将 dict 映射为长度为 n_features 的向量（只取单特征 Shapley）
                shap_vec = np.zeros(len(raw_cols), dtype=np.float64)
                for k, v in sv.dict_values.items():
                    if isinstance(k, tuple) and len(k) == 1:
                        shap_vec[k[0]] = v
                abs_shap_sum += np.abs(shap_vec)
            except ValueError as e:
                # 兼容“全常数将被删除”的错误：跳过该样本
                if "All features are constant" in str(e):
                    skipped += 1
                    print(f"[warn] 样本 {i+1}/{n_eval} 遇到常数子集，已跳过。", flush=True)
                else:
                    raise
            finally:
                dt = perf_counter() - t1
                avg_t = dt if avg_t is None else (0.9 * avg_t + 0.1 * dt)
                remain = (n_eval - (i + 1)) * (avg_t if avg_t else dt)
                pbar.update(1)
                pbar.set_postfix(last_s=f"{dt:.2f}", avg_s=f"{avg_t:.2f}", eta_s=f"{remain:.0f}")
                if SAVE_EVERY and (i + 1) % SAVE_EVERY == 0:
                    save_partial(i + 1 - skipped)

    # 汇总与保存最终结果（用有效样本数）
    effective = max(1, n_eval - skipped)
    mean_abs_shap = abs_shap_sum / effective
    imp_df = pd.DataFrame({"feature": raw_cols, "mean_abs_shap": mean_abs_shap})
    imp_df = imp_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print(f"\n[SHAP · RAW] Top-{TOPK_PRINT} features (mean |SHAP| over {effective} effective samples):")
    print(imp_df.head(TOPK_PRINT).to_string(index=False))

    imp_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[done] SHAP完成 · 用时 {(perf_counter()-t):.2f}s · 已保存：{os.path.abspath(OUT_CSV)}")

    # 总耗时
    section("全部完成")
    print(f"总用时：{(perf_counter()-t0):.2f}s")
    if os.path.exists(OUT_CSV_PARTIAL):
        print(f"[info] 中间结果（增量保存）在：{os.path.abspath(OUT_CSV_PARTIAL)}")


if __name__ == "__main__":
    main()


c:\anaconda\envs\mmhf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



============ 数据读取与清洗 ============
[warn] 发现未映射到的标签值（将被剔除）：
Group
AD    219
CN    204
[hint] 如需包含它们，请调整 CLASSES 或预先过滤。
[info] 因标签未映射/缺失删除样本数：423
[info] 过滤零方差特征 2 列：['PXABNORM', 'NXABNORM']
[done] 用时 0.04s · 数据形状: X=(479, 130), y=479

============ 划分训练/测试集 ============
Train: (383, 130), Test: (96, 130), Pos rate(train)=0.329
[done] 用时 0.00s

============ 训练 TabPFN ============


RuntimeError: Failed to download TabPFN v2.5 model 'tabpfn-v2.5-classifier-v2.5_default.ckpt'.

Details and instructions:
HuggingFace authentication error downloading from 'Prior-Labs/tabpfn_2_5'.
This model is gated and requires you to accept its terms.

Please follow these steps:
1. Visit https://huggingface.co/Prior-Labs/tabpfn_2_5 in your browser and accept the terms of use.
2. Log in to your Hugging Face account via the command line by running:
   hf auth login
   (Alternatively, you can set the HF_TOKEN environment variable   with a read token.)

For detailed instructions, see https://docs.priorlabs.ai/how-to-access-gated-models

For commercial usage, we provide alternative download options for TabPFN v2.5; please reach out to us at sales@priorlabs.ai.